# dl_helper Kaggle 真实发布门禁

按顺序运行所有代码单元。本门禁验证安装、配置、自动预检与一次完整训练成功（退出码 0）。75（PREEMPTED）在极短调试运行中不应出现；若出现则按「成功暂停」处理而非失败。Secret 只从 Kaggle Secrets 读取。

In [ ]:
import os

os.environ['DL_HELPER_GIT_REPO'] = 'https://github.com/lhiqwj173/dl_helper.git'
# 模板测试标记：0000000000000000000000000000000000000000 仅用于识别占位值，禁止实际使用。
os.environ['DL_HELPER_GIT_REF'] = 'f193ecf73203ba62b4cead62b6f684820c328c10'
os.environ['DL_HELPER_MNIST_PATH'] = '/kaggle/input/datasets/vikramtiwari/mnist-numpy/mnist.npz'
os.environ['DL_HELPER_RUN_ID'] = '-'.join(['mnist', 'release', 'gate', 'parent', 'publish', '20260810'])
os.environ['ALIST_HOST'] = 'https://tmsatws.kdns.fr'
os.environ['ALIST_BASE_PATH'] = '/dl-helper/release-gate'
os.environ['WECOM_TO_USER'] = '@all'
print('代码版本、MNIST 路径和运行时服务地址已设置')

In [ ]:
import os
import subprocess
import sys

repo_dir = '/kaggle/working/dl-helper'
if os.path.exists(repo_dir):
    raise RuntimeError(f'目录已存在，请新建 Kaggle Session 后重试: {repo_dir}')

def checked(argv, *, cwd=None):
    proc = subprocess.run(argv, cwd=cwd, capture_output=True, text=True, encoding='utf-8')
    if proc.stdout:
        print(proc.stdout, end='')
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr, file=sys.stderr, end='')
        raise SystemExit(proc.returncode)
    return proc

checked(['git', 'clone', os.environ['DL_HELPER_GIT_REPO'], repo_dir])
checked(['git', 'checkout', os.environ['DL_HELPER_GIT_REF']], cwd=repo_dir)
head = checked(['git', 'rev-parse', 'HEAD'], cwd=repo_dir).stdout.strip()
if head.lower() != os.environ['DL_HELPER_GIT_REF'].lower():
    raise RuntimeError(f'checkout HEAD 不匹配: {head}')
os.environ['DL_HELPER_REPO_DIR'] = repo_dir
print(f'[bootstrap] fixed revision: {head}')

In [ ]:
import subprocess
import sys
proc = subprocess.run([sys.executable, '/kaggle/working/dl-helper/envs/kaggle_bootstrap.py'], capture_output=True, text=True, encoding='utf-8')
print(proc.stdout, end='')
if proc.returncode != 0:
    print(proc.stderr, file=sys.stderr, end='')
    raise SystemExit(proc.returncode)
print('[bootstrap] OK')

In [ ]:
from pathlib import Path
import yaml

config_path = Path('/kaggle/working/dl-helper-kaggle.yaml')
source_config_path = Path('/kaggle/working/dl-helper/examples/configs/kaggle/mnist.yaml')
with source_config_path.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
config['remote'] = {'type': 'alist', 'host': os.environ['ALIST_HOST'], 'base_path': os.environ['ALIST_BASE_PATH'], 'user_secret_key': 'ALIST_USER', 'password_secret_key': 'ALIST_PWD', 'connect_timeout_seconds': 10, 'read_timeout_seconds': 60, 'max_attempts': 3, 'async_upload': False, 'failure_policy': 'required'}
config['notifications'] = {'type': 'wecom', 'corp_id_secret_key': 'WECOM_CORP_ID', 'corp_secret_key': 'WECOM_CORP_SECRET', 'agent_id_secret_key': 'WECOM_AGENT_ID', 'to_user': os.environ['WECOM_TO_USER'], 'connect_timeout_seconds': 10, 'read_timeout_seconds': 30, 'max_attempts': 3, 'failure_policy': 'required'}
config['run']['id'] = os.environ['DL_HELPER_RUN_ID']
config['run']['source_revision'] = os.environ['DL_HELPER_GIT_REF']
config['backend']['torch']['mixed_precision'] = 'no'
# D-003：Kaggle 预算由平台执行策略固定为 660/10，配置不再包含 runtime。
with config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)
print(f'配置已写入: {config_path}')
print('AList host:', config['remote']['host'])
print('mixed precision:', config['backend']['torch']['mixed_precision'])
print('run ID:', config['run']['id'])

In [ ]:
import requests
response = requests.get('https://tmsatws.kdns.fr/api/public/settings', timeout=20)
print('HTTPS status:', response.status_code)
if response.status_code != 200:
    raise RuntimeError(f'AList HTTPS 检查失败: HTTP {response.status_code}')

In [ ]:
import subprocess, sys
project_dir = '/kaggle/working/dl-helper/examples'
proc = subprocess.run([sys.executable, '-m', 'dl_helper.training.cli', 'train', '--config', str(config_path), '--project-dir', project_dir, '--experiment', 'experiments.mnist:build_experiment', '--preflight-only'], cwd='/kaggle/working/dl-helper', text=True, encoding='utf-8')
print('preflight exit code:', proc.returncode)
if proc.returncode != 0:
    raise SystemExit(proc.returncode)

## 训练

运行下一个单元执行一次完整训练，预期 `train exit code: 0`。若出现 `75` 表示预算保护暂停（同样已保存检查点），不作为失败。

In [ ]:
import subprocess, sys
project_dir = '/kaggle/working/dl-helper/examples'
proc = subprocess.run([sys.executable, '-m', 'dl_helper.training.cli', 'train', '--config', str(config_path), '--project-dir', project_dir, '--experiment', 'experiments.mnist:build_experiment', '--run-id', os.environ['DL_HELPER_RUN_ID']], cwd='/kaggle/working/dl-helper', text=True, encoding='utf-8')
print('train exit code:', proc.returncode)
if proc.returncode not in (0, 75):
    raise RuntimeError(f'训练失败，退出码 {proc.returncode}')

## 成功终态幂等性

同一 `--run-id` 不可改写：已完成 run 再次启动必须被拒绝（非零），验证成功终态互斥。

In [ ]:
import subprocess, sys
project_dir = '/kaggle/working/dl-helper/examples'
proc = subprocess.run([sys.executable, '-m', 'dl_helper.training.cli', 'train', '--config', str(config_path), '--project-dir', project_dir, '--experiment', 'experiments.mnist:build_experiment', '--run-id', os.environ['DL_HELPER_RUN_ID']], cwd='/kaggle/working/dl-helper', text=True, encoding='utf-8')
print('rerun exit code:', proc.returncode)
if proc.returncode == 0:
    raise RuntimeError('已完成 run 被意外改写，幂等性失效')

In [ ]:
from pathlib import Path
import hashlib
run_dir = Path('/kaggle/working/dl-helper-runs/runs') / os.environ['DL_HELPER_RUN_ID']
required = [run_dir / 'run-manifest.json', run_dir / 'services' / 'service-manifest.json', run_dir / 'services' / 'service-audit.jsonl', run_dir / 'report' / 'index.html']
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'缺少发布工件: {path}')
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(path.relative_to(run_dir), digest)
print('唯一成功终态和服务审计工件已找到')